In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
%pip install kagglehub
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import random
import re
import cv2
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf

# --- VRAM FIX: Enable Memory Growth ---
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Memory growth enabled on {len(gpus)} GPUs")
    except RuntimeError as e:
        print(e)
# --------------------------------------

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score
from sklearn.model_selection import train_test_split
from tensorflow.keras import callbacks, layers, metrics
from concurrent.futures import ThreadPoolExecutor

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

try:
    from tensorflow.keras.applications import EfficientNetV2S as Backbone
    from tensorflow.keras.applications.efficientnet_v2 import preprocess_input
except Exception:
    from tensorflow.keras.applications import EfficientNetB4 as Backbone
    from tensorflow.keras.applications.efficientnet import preprocess_input

%matplotlib inline

# ---------------------------------------------------------------------------
# Config  (Tuned for >90% Accuracy under 1 Hour)
# ---------------------------------------------------------------------------
SEED = 42
IMG_SIZE = 384
BATCH_SIZE = 16
HEAD_EPOCHS = 4
FINETUNE_EPOCHS = 18
NUM_WORKERS = os.cpu_count() or 4 # Corrected to call os.cpu_count()

CACHE_DIR = os.path.abspath("odir_cache_384_multilabel")
CKPT_PATH = os.path.abspath(os.path.join("checkpoints", "best_fundus_90.keras"))

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Corrected strategy initialization based on number of GPUs
gpus = tf.config.list_physical_devices('GPU')
if len(gpus) > 1:
    strategy = tf.distribute.MirroredStrategy()
    print(f"Number of devices: {strategy.num_replicas_in_sync} (MirroredStrategy)")
elif len(gpus) == 1:
    strategy = tf.distribute.OneDeviceStrategy(device="/gpu:0")
    print("Number of devices: 1 (OneDeviceStrategy)")
else:
    strategy = tf.distribute.OneDeviceStrategy(device="/cpu:0")
    print("Number of devices: 1 (OneDeviceStrategy - CPU)")

try:
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
    print("Mixed precision on")
except Exception as exc:
    print("Mixed precision not set:", exc)

Memory growth enabled on 1 GPUs
Number of devices: 1 (OneDeviceStrategy)
Mixed precision on


This block imports required libraries, enables memory growth for TensorFlow to prevent it from unnecessarily hoarding all VRAM, and attempts to load EfficientNetV2S (falling back to EfficientNetB4 if unavailable). It also sets deterministic random seeds for reproducibility and configures global hyperparameters.

Alternative Approach: A configuration file (like YAML or JSON) or standard Python argparse to handle hyperparameters dynamically, rather than hardcoding them into the script.

Reasoning for Current Approach: Hardcoding configurations at the top of a single script is the most practical and frictionless approach for Kaggle notebooks or standalone executable scripts, making the code entirely self-contained and easy to modify on the fly.

In [ ]:
# Disease Regex for Multi-Label Parsing
DISEASE_RE = {
    "D": re.compile(r"retinopath|diabetic|macular edema|laser spot|cotton wool|proliferative|nonproliferative|non proliferative", re.I),
    "G": re.compile(r"glaucoma|optic (disk|disc) cupping|cup-to-disc", re.I),
    "C": re.compile(r"cataract", re.I),
    "A": re.compile(r"macular degeneration|\bamd\b|geographic atrophy", re.I),
    "H": re.compile(r"hypertens", re.I),
    "M": re.compile(r"pathological myopia|myopic maculopathy|\bmyopia\b", re.I),
}
NORMAL_RE = re.compile(r"^normal fundus$", re.I)

try:
    AdamW = tf.keras.optimizers.AdamW
except AttributeError:
    AdamW = tf.keras.optimizers.Adam

def split_keywords(text) -> list[str]:
    raw = str(text).strip().lower()
    if raw in {"", "nan", "none"}:
        return []
    raw = raw.replace("\uff0c", ",").replace("，", ",")
    return [re.sub(r"\s+", " ", p).strip(" .;") for p in raw.split(",") if p.strip()]

def get_multilabel(keywords) -> np.ndarray | None:
    """Parses keywords into a one-hot vector: [N, D, G, C, A, H, M, O]"""
    parts = split_keywords(keywords)
    if not parts:
        return None
    blob = ", ".join(parts)
    if re.search(r"low image quality", blob):
        return None

    ignorable = {"lens dust", "optic disk photographically invisible", "image offset", "anterior segment image", "no fundus image"}
    meaningful = [p for p in parts if p not in ignorable]
    if not meaningful:
        return None

    vec = np.zeros(8, dtype=np.float32)
    has_normal = False
    disease_found = False

    for p in meaningful:
        if NORMAL_RE.match(p):
            has_normal = True
            continue
        mapped = False
        if DISEASE_RE["D"].search(p): vec[1] = 1.0; mapped = True; disease_found = True
        if DISEASE_RE["G"].search(p): vec[2] = 1.0; mapped = True; disease_found = True
        if DISEASE_RE["C"].search(p): vec[3] = 1.0; mapped = True; disease_found = True
        if DISEASE_RE["A"].search(p): vec[4] = 1.0; mapped = True; disease_found = True
        if DISEASE_RE["H"].search(p): vec[5] = 1.0; mapped = True; disease_found = True
        if DISEASE_RE["M"].search(p): vec[6] = 1.0; mapped = True; disease_found = True
        if not mapped:
            vec[7] = 1.0 # Other
            disease_found = True

    if has_normal and not disease_found:
        vec[0] = 1.0

    if vec.sum() == 0:
        vec[7] = 1.0 # Default to Other if completely unrecognized

    return vec

def keywords_for_image(row, image_name) -> str:
    name = str(image_name).lower()
    if "left" in name: return row.get("Left-Diagnostic Keywords", "")
    if "right" in name: return row.get("Right-Diagnostic Keywords", "")
    return ""

def collect_rows(df: pd.DataFrame) -> pd.DataFrame:
    records = []
    if "filename" in df.columns:
        for _, row in df.iterrows():
            image_name = row["filename"]
            kw = keywords_for_image(row, image_name)
            vec = get_multilabel(kw)
            if vec is not None:
                records.append({
                    "Patient_ID": row["ID"] if "ID" in row.index else row.get("Patient Id"),
                    "Image_Name": image_name,
                    "Label": vec,
                })
    else:
        patient_df = df.drop_duplicates(subset=["ID"])
        for _, row in patient_df.iterrows():
            for image_name, kw in [
                (row["Left-Fundus"], row["Left-Diagnostic Keywords"]),
                (row["Right-Fundus"], row["Right-Diagnostic Keywords"]),
            ]:
                vec = get_multilabel(kw)
                if vec is not None:
                    records.append({
                        "Patient_ID": row["ID"],
                        "Image_Name": image_name,
                        "Label": vec,
                    })

    out = pd.DataFrame(records).dropna(subset=["Image_Name", "Patient_ID"])
    out = out.drop_duplicates(subset=["Image_Name"])
    return out.reset_index(drop=True)

The ODIR dataset uses free-text diagnostic keywords.
This segment utilizes Regular Expressions (Regex) to safely parse these raw strings, ignore low-quality images, and map specific diseases (like diabetes, glaucoma, cataracts) to a definitive "DISEASE" or "N" (Normal) category. collect_rows consolidates this into a clean binary label dataframe.

Alternative Approach: If the dataset metadata includes pre-processed one-hot encoded boolean columns (like N, D, G), those could be mapped directly. Alternatively, a lightweight NLP model to classify the text strings can be used.

Reasoning for Current Approach: Raw text parsing via Regex is highly robust against missing or malformed one-hot columns in different versions of the ODIR dataset. It guarantees exact control over what constitutes a "defective" eye based purely on clinical notes.

In [ ]:
def crop_fundus(img, tol=7):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    mask = gray > tol
    if not np.any(mask):
        return img
    img = img[np.ix_(mask.any(1), mask.any(0))]
    h, w = img.shape[:2]
    side = max(h, w)
    y0, x0 = (side - h) // 2, (side - w) // 2
    canvas = np.zeros((side, side, 3), dtype=img.dtype)
    canvas[y0 : y0 + h, x0 : x0 + w] = img
    return canvas

def load_fundus_uint8(path, size=IMG_SIZE):
    img = cv2.imread(path)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = crop_fundus(img)
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l_ch, a_ch, b_ch = cv2.split(lab)
    l_ch = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(l_ch)
    img = cv2.cvtColor(cv2.merge((l_ch, a_ch, b_ch)), cv2.COLOR_LAB2RGB)
    return cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)

def _process_and_save(item):
    src, dst, label_vec = item
    if not os.path.isfile(dst):
        img = load_fundus_uint8(src)
        if img is not None:
            cv2.imwrite(dst, cv2.cvtColor(img, cv2.COLOR_RGB2BGR), [int(cv2.IMWRITE_JPEG_QUALITY), 95])
            return dst, label_vec
    elif os.path.isfile(dst):
        return dst, label_vec
    return None

def cache_split(frame, image_dir, split_name):
    tasks = []
    out_dir = os.path.join(CACHE_DIR, split_name)
    os.makedirs(out_dir, exist_ok=True)

    for _, row in frame.iterrows():
        src = os.path.join(image_dir, str(row["Image_Name"]))
        dst = os.path.join(out_dir, os.path.splitext(str(row["Image_Name"]))[0] + ".jpg")
        tasks.append((src, dst, row["Label"]))

    paths, labels = [], []
    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
        results = executor.map(_process_and_save, tasks)
        for res in results:
            if res is not None:
                paths.append(res[0])
                labels.append(res[1])

    print(f"{split_name}: {len(paths)} cached images at {IMG_SIZE}x{IMG_SIZE}")
    return np.array(paths), np.stack(labels).astype(np.float32)

Fundus images have large, uninformative black borders. crop_fundus thresholds the image to find the actual eye and crops out dead space. load_fundus_uint8 applies CLAHE (Contrast Limited Adaptive Histogram Equalization) on the lightness channel to enhance blood vessels. Finally, cache_split uses Python's ThreadPoolExecutor to process and save these cleaned images to a temporary disk folder.

Alternative Approach: Image cropping and CLAHE could be applied dynamically during training inside the tf.data pipeline, rather than caching them to disk beforehand.

Reasoning for Current Approach: Caching preprocessed images to disk is incredibly efficient. Performing heavy CPU-bound operations like CLAHE and masking on-the-fly during training often starves the GPU of data, creating a massive bottleneck. Pre-caching ensures the GPU runs at maximum utilization.

In [ ]:
def make_dataset(paths, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(min(len(paths), 5000), seed=SEED, reshuffle_each_iteration=True)

    def _load(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        img = tf.cast(img, tf.float32)
        return img, label

    def _aug(img, label):
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        img = tf.image.rot90(img, tf.random.uniform([], 0, 4, dtype=tf.int32))
        zoom = tf.random.uniform([], 0.80, 1.0)
        side = tf.cast(tf.cast(IMG_SIZE, tf.float32) * zoom, tf.int32)
        img = tf.image.random_crop(img, [side, side, 3])
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        img = tf.image.random_brightness(img, 0.1)
        img = tf.image.random_contrast(img, 0.85, 1.15)
        img = tf.clip_by_value(img, 0.0, 255.0)
        return img, label

    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(_aug, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda img, y: (preprocess_input(img), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

def compile_model(model, lr):
    model.compile(
        optimizer=AdamW(learning_rate=lr, weight_decay=1e-4) if AdamW is tf.keras.optimizers.AdamW else AdamW(learning_rate=lr),
        loss=tf.keras.losses.BinaryFocalCrossentropy(gamma=2.0, alpha=0.25, label_smoothing=0.05), # THE SECRET SAUCE
        metrics=[
            metrics.AUC(name="auc", multi_label=True),
        ],
    )


This constructs the TensorFlow tf.data dataset pipeline. It handles parallel data loading and applies dynamic, randomized data augmentations (flips, rotations, slight zooming/cropping, brightness/contrast changes) natively using TensorFlow operations.

Alternative Approach: A standard ImageDataGenerator (legacy) or external libraries like Albumentations or Keras CV for potentially more complex, medically specific augmentations.

Reasoning for Current Approach: The tf.data API is the absolute fastest way to feed a Keras model, as it natively supports multi-threading, vectorization, and prefetching. Using raw TF ops for augmentation allows the entire pipeline to remain within the TensorFlow graph, preventing CPU-to-GPU memory transfer slowdowns.

In [ ]:
def build_model():
    base = Backbone(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(8, activation="sigmoid", dtype="float32")(x) # 8 Classes now!
    model = tf.keras.Model(inputs, outputs)
    compile_model(model, lr=1e-3)
    return model, base

def unfreeze_except_bn(base, frac=0.8):
    base.trainable = True
    freeze_until = int(len(base.layers) * (1.0 - frac))
    for i, layer in enumerate(base.layers):
        is_bn = isinstance(layer, layers.BatchNormalization)
        layer.trainable = (i >= freeze_until) and (not is_bn)


This segment builds the neural network classifier. It instantiates an EfficientNet backbone pre-trained on ImageNet, adds a custom classification head (GlobalAveragePooling, Dropout, Dense layers), and outputs a binary probability. unfreeze_except_bn is a utility function used in Phase 2 to unfreeze the deeper network weights while keeping Batch Normalization layers frozen.

Alternative Approach: Using a Vision Transformer (ViT) or ResNet architecture. Additionally, unfreezing the entire network immediately instead of doing a two-phase approach.

Reasoning for Current Approach: EfficientNet offers state-of-the-art accuracy with significantly fewer parameters than older models like ResNet. The two-phase training (training the head first, then fine-tuning) prevents random initialized weights from wrecking the pre-trained backbone. Keeping Batch Normalization frozen during fine-tuning is a critical MLOps practice to maintain stable training statistics.

In [ ]:
def tta_probs(model, paths):
    def _load_eval(path):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        img = tf.cast(img, tf.float32)
        orig = preprocess_input(img)
        hflip = preprocess_input(tf.image.flip_left_right(img))
        vflip = preprocess_input(tf.image.flip_up_down(img))
        return tf.stack([orig, hflip, vflip])

    ds = tf.data.Dataset.from_tensor_slices(paths)
    ds = ds.map(_load_eval, num_parallel_calls=tf.data.AUTOTUNE)

    eval_batch_size = max(1, BATCH_SIZE // 3)
    ds = ds.batch(eval_batch_size).prefetch(tf.data.AUTOTUNE)

    all_preds = []
    for batch in ds:
        b_size = tf.shape(batch)[0]
        flattened = tf.reshape(batch, [-1, IMG_SIZE, IMG_SIZE, 3])
        preds = model(flattened, training=False)
        preds = tf.reshape(preds, [b_size, 3, 8]) # Now 8 classes
        mean_preds = tf.reduce_mean(preds, axis=1)
        all_preds.append(mean_preds.numpy())

    return np.concatenate(all_preds, axis=0)

def best_accuracy_threshold(y_true, y_prob):
    best_t, best_a = 0.5, 0.0
    for t in np.linspace(0.20, 0.80, 61):
        acc = float(np.mean((y_prob >= t).astype(int) == y_true))
        if acc > best_a:
            best_a, best_t = acc, float(t)
    return best_t, best_a

TTA generates predictions for the original image as well as horizontally and vertically flipped versions, averaging the results for a final prediction. The threshold tuner function scans probability cutoffs (from 0.2 to 0.8) on the validation set to find the optimal threshold for separating healthy vs. defective, rather than blindly using 0.5.

Alternative Approach: Perform a standard single-pass inference without TTA and rely entirely on the default 0.5 threshold.

Reasoning for Current Approach: TTA acts as a "free" ensemble method, providing more robust and confident predictions on borderline cases. Tuning the threshold is essential for medical datasets, which are often imbalanced; a 0.5 threshold rarely yields the optimal balance of sensitivity and specificity.

In [ ]:
dataset_path = kagglehub.dataset_download(
    "andrewmvd/ocular-disease-recognition-odir5k"
)

print(dataset_path)

for root, dirs, files in os.walk(dataset_path):
    print("\nROOT:", root)
    print("FILES:", files[:10])
    print("DIRS:", dirs[:10])

Using Colab cache for faster access to the 'ocular-disease-recognition-odir5k' dataset.
/kaggle/input/ocular-disease-recognition-odir5k

ROOT: /kaggle/input/ocular-disease-recognition-odir5k
FILES: ['full_df.csv']
DIRS: ['preprocessed_images', 'ODIR-5K']

ROOT: /kaggle/input/ocular-disease-recognition-odir5k/preprocessed_images
FILES: ['3419_left.jpg', '4176_right.jpg', '3370_left.jpg', '1255_right.jpg', '660_left.jpg', '484_right.jpg', '4221_right.jpg', '2396_left.jpg', '543_left.jpg', '3017_left.jpg']
DIRS: []

ROOT: /kaggle/input/ocular-disease-recognition-odir5k/ODIR-5K
FILES: []
DIRS: ['ODIR-5K']

ROOT: /kaggle/input/ocular-disease-recognition-odir5k/ODIR-5K/ODIR-5K
FILES: ['data.xlsx']
DIRS: ['Testing Images', 'Training Images']

ROOT: /kaggle/input/ocular-disease-recognition-odir5k/ODIR-5K/ODIR-5K/Testing Images
FILES: ['3552_right.jpg', '4711_left.jpg', '3534_right.jpg', '1911_right.jpg', '4691_left.jpg', '3588_right.jpg', '1701_right.jpg', '4739_right.jpg', '1343_right.jpg', '

In [12]:
def main() -> None:
    print("=" * 72)
    print("CHAMPIONSHIP TRAINING: Multi-Label Focal Loss Strategy")
    print("=" * 72)

####
    dataset_path = kagglehub.dataset_download(
        "andrewmvd/ocular-disease-recognition-odir5k"
    )

    print("Using dataset:", dataset_path)

    csv_path = os.path.join(dataset_path, "full_df.csv")
    image_dir = os.path.join(dataset_path, "preprocessed_images")

    if not os.path.isfile(csv_path):
        raise FileNotFoundError(
            f"full_df.csv not found at: {csv_path}"
        )

    if not os.path.isdir(image_dir):
        raise FileNotFoundError(
            f"preprocessed_images not found at: {image_dir}"
        )

    print("CSV:", csv_path)
    print("Images:", image_dir)

  ####

    df = pd.read_csv(csv_path)
    table = collect_rows(df)
    table["Exists"] = table["Image_Name"].map(lambda n: os.path.isfile(os.path.join(image_dir, str(n))))
    table = table[table["Exists"]].drop(columns=["Exists"])

    # Stratify strictly by checking if ANY disease is present
    patient_y = table.groupby("Patient_ID")["Label"].apply(lambda x: np.max(np.vstack(x), axis=0))
    pids, ys = patient_y.index.to_numpy(), np.vstack(patient_y.to_numpy())
    is_diseased = np.max(ys[:, 1:], axis=1) # 1 if any disease class is positive

    train_pts, temp_pts, _, temp_y = train_test_split(pids, is_diseased, test_size=0.30, random_state=SEED, stratify=is_diseased)
    val_pts, test_pts = train_test_split(temp_pts, test_size=0.50, random_state=SEED, stratify=temp_y)

    train_df = table[table["Patient_ID"].isin(train_pts)]
    val_df = table[table["Patient_ID"].isin(val_pts)]
    test_df = table[table["Patient_ID"].isin(test_pts)]

    print(f"Images train/val/test: {len(train_df)} / {len(val_df)} / {len(test_df)}")

    print(f"Caching MULTI-LABEL images across {NUM_WORKERS} CPU workers...")
    train_p, train_y = cache_split(train_df, image_dir, "train")
    val_p, val_y = cache_split(val_df, image_dir, "val")
    test_p, test_y = cache_split(test_df, image_dir, "test")

    train_ds = make_dataset(train_p, train_y, training=True)
    val_ds = make_dataset(val_p, val_y, training=False)

    os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)

    # Notice: No class_weights dictionary needed because Focal Loss inherently balances classes
    cbs_phase1 = [
        callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.3, patience=2, min_lr=6.4920e-06, verbose=1),
        callbacks.ModelCheckpoint(CKPT_PATH, monitor="val_auc", mode="max", save_best_only=True, verbose=1),
    ]

    cbs_phase2 = [
        callbacks.ModelCheckpoint(CKPT_PATH, monitor="val_auc", mode="max", save_best_only=True, verbose=1),
    ]

    tf.keras.backend.clear_session()

    with strategy.scope():
        model, base = build_model()

    print("Phase 1: train head, backbone frozen")
    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=HEAD_EPOCHS,
        callbacks=cbs_phase1,
        verbose=1,
    )

    print("Phase 2: fine-tune top 80% of backbone")
    with strategy.scope():
        unfreeze_except_bn(base, frac=0.8)
        steps = max(1, (len(train_p) // BATCH_SIZE) * FINETUNE_EPOCHS)
        cosine = tf.keras.optimizers.schedules.CosineDecay(5e-5, steps, alpha=0.12984)
        compile_model(model, lr=cosine)

    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=FINETUNE_EPOCHS,
        callbacks=cbs_phase2,
        verbose=1,
    )

    if os.path.isfile(CKPT_PATH):
        model = tf.keras.models.load_model(CKPT_PATH)

    print("Tuning threshold on validation (Fast TTA)...")
    val_prob = tta_probs(model, val_p) # Returns shape (N, 8)

    # Convert Multi-Label back to Binary for Competition Metrics
    # Probability of being defective = Max probability of ANY disease class (Columns 1 through 7)
    val_y_binary = np.max(val_y[:, 1:], axis=1).astype(int)
    val_prob_binary = np.max(val_prob[:, 1:], axis=1)

    thr, val_acc = best_accuracy_threshold(val_y_binary, val_prob_binary)
    print(f"Val binary accuracy @ tuned threshold {thr:.3f}: {val_acc:.4f} AUC={roc_auc_score(val_y_binary, val_prob_binary):.4f}")

    print("Evaluating test set (Fast TTA)...")
    test_prob = tta_probs(model, test_p)

    # Calculate Binary Vectors for Test Data
    y_true_binary = np.max(test_y[:, 1:], axis=1).astype(int)
    test_prob_binary = np.max(test_prob[:, 1:], axis=1)
    y_pred_binary = (test_prob_binary >= thr).astype(int)

    # Calculate Core Metrics
    test_acc = float(np.mean(y_pred_binary == y_true_binary))
    roc_auc = roc_auc_score(y_true_binary, test_prob_binary)
    f1 = f1_score(y_true_binary, y_pred_binary)

    # Calculate Confusion Matrix, Sensitivity, and Specificity
    cm = confusion_matrix(y_true_binary, y_pred_binary)
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    print("\n" + "=" * 72)
    print("FINAL PERFORMANCE METRICS (Competition Binary Task)")
    print("=" * 72)
    print(f"ROC-AUC (Primary):       {roc_auc:.4f}")
    print(f"Accuracy:                {test_acc:.4f} ({test_acc:.1%})")
    print(f"F1 Score:                {f1:.4f}")
    print(f"Sensitivity (Recall):    {sensitivity:.4f}")
    print(f"Specificity:             {specificity:.4f}")
    print("-" * 72)
    print("CONFUSION MATRIX:")
    print(f"                 Predicted Healthy (0)   Predicted Defective (1)")
    print(f"Actual Healthy (0)        {tn:<22} {fp}")
    print(f"Actual Defect. (1)        {fn:<22} {tp}")
    print("=" * 72)

if __name__ == "__main__":
    main()

CHAMPIONSHIP TRAINING: Multi-Label Focal Loss Strategy
Using Colab cache for faster access to the 'ocular-disease-recognition-odir5k' dataset.
Using dataset: /kaggle/input/ocular-disease-recognition-odir5k
CSV: /kaggle/input/ocular-disease-recognition-odir5k/full_df.csv
Images: /kaggle/input/ocular-disease-recognition-odir5k/preprocessed_images
Images train/val/test: 4451 / 971 / 969
Caching MULTI-LABEL images across 2 CPU workers...
train: 4451 cached images at 384x384
val: 971 cached images at 384x384
test: 969 cached images at 384x384
82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Phase 1: train head, backbone frozen
Epoch 1/4
279/279 ━━━━━━━━━━━━━━━━━━━━ 0s 236ms/step - auc: 0.6306 - loss: 0.0951
Epoch 1: val_auc improved from None to 0.81222, saving model to /content/checkpoints/best_fundus_90.keras

Epoch 1: finished saving model to /content/checkpoints/best_fundus_90.keras
279/279 ━━━━━━━━━━━━━━━━━━━━ 106s 317ms/step - auc: 0.6821 - loss: 0.0838 - val_auc: 0.8122 - val_loss:

It handles dataset fetching, groups data by Patient_ID, performs a stratified split, computes class weights to combat imbalance, runs the two-phase training loop with callbacks (checkpointing, LR decay), and finally prints out clinically relevant metrics (ROC-AUC, Sensitivity, Specificity) and a confusion matrix.

Alternative Approach: Instead of a single Train/Val/Test split,  K-Fold Cross-Validation could be implemented. The images could be randomly split without grouping them by patient.

Reasoning for Current Approach: Splitting strictly by Patient_ID is the most crucial part of this block; it prevents severe data leakage (e.g., having a patient's left eye in the training set and right eye in the test set), ensuring realistic evaluation. Class weights ensure the model doesn't just blindly guess the majority class, and cosine learning rate decay allows for smooth convergence during fine-tuning.